# Set Covering Problem — Constructive Heuristics
### Exercise 1.1 — Data reading & instance representation

This notebook is the foundation for the assignment *"Constructive heuristics for the SCP"*.

In this first part we build:
1. A parser for the OR-Library SCP instance format.
2. An `SCPInstance` class that stores the problem and exposes the incremental
   bookkeeping (covered elements, coverage counts, candidate columns, cost)
   that CH1–CH3 and the redundancy-elimination procedure will need later.
3. A small sanity check loading the two example instances (`scp57.txt`,
   `scp58.txt`) and comparing basic stats against the best-known solutions
   table given in the assignment.

Later notebooks/sections will add: CH1, CH2, CH3, redundancy elimination (RE),
and the experimental comparison (average % deviation from BKS, fraction of
instances improved by RE).

## 1. The OR-Library SCP file format

Each instance file is a whitespace-separated stream of integers, laid out as:

```
m n
c_1 c_2 ... c_n              # cost of every column (set), n values
k_1 <k_1 column indices>     # row 1: how many columns cover it, then which ones (1-based)
k_2 <k_2 column indices>     # row 2
...
k_m <k_m column indices>     # row m
```

- `m` = number of elements (rows) that must be covered
- `n` = number of available sets (columns)
- Column indices are **1-based** in the file.


In [2]:
import numpy as np
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Set, Dict
import random

DATA_DIR = Path("SCP-Instances")

## 2. Parser

In [3]:
def read_scp_instance(filepath):
    """Parse an OR-Library formatted SCP instance file.

    Returns dict with:
      m, n    : ints
      costs   : np.ndarray (n,)
      rows    : list[list[int]] length m -- 0-indexed sets covering each element
      cols    : list[list[int]] length n -- 0-indexed elements covered by each set
    """
    with open(filepath, "r") as f:
        tokens = list(map(int, f.read().split()))

    idx = 0
    m, n = tokens[idx], tokens[idx + 1]
    idx += 2

    costs = np.array(tokens[idx: idx + n], dtype=float)
    idx += n

    rows = []
    for _ in range(m):
        k = tokens[idx]; idx += 1
        rows.append([c - 1 for c in tokens[idx: idx + k]])
        idx += k

    assert idx == len(tokens), (
        f"{filepath}: parser consumed {idx} of {len(tokens)} tokens -- "
        "file does not match the expected OR-Library SCP format"
    )

    cols = [[] for _ in range(n)]
    for row_idx, covering_sets in enumerate(rows):
        for s in covering_sets:
            cols[s].append(row_idx)

    return {"m": m, "n": n, "costs": costs, "rows": rows, "cols": cols}

## 3. `SCPInstance` — problem wrapper with incremental state

This class holds the static problem data plus the *mutable* state of a
partial solution under construction:

- `covered`: boolean mask of covered elements
- `cover_count`: how many chosen subsets currently cover each element (needed to
  detect, during redundancy elimination, whether removing a subset leaves some
  element uncovered)
- `chosen`: boolean mask of subsets already in the solution
- `remaining_gain(j)`: number of *currently uncovered* elements that subset `j`
  would newly cover — this is exactly the quantity CH1–CH3 need to (a) decide
  whether a subset is redundant (`remaining_gain == 0` ⇒ skip it) and (b) drive
  greedy choices in later heuristics, all updated incrementally instead of
  recomputed from scratch at every step.

In [4]:
class SCPInstance:
    def __init__(self, data, name=None):
        self.name = name
        self.m, self.n = data["m"], data["n"]
        self.costs = data["costs"]
        self.rows = data["rows"]   # element -> covering subsets
        self.cols = data["cols"]   # set -> covered elements
        self.reset()

    def reset(self):
        self.covered = np.zeros(self.m, dtype=bool)
        self.cover_count = np.zeros(self.m, dtype=int)
        self.chosen = np.zeros(self.n, dtype=bool)
        self.num_covered = 0
        self.solution_cost = 0.0

    def uncovered_elements(self):
        return np.flatnonzero(~self.covered)

    def remaining_gain(self, j):
        if self.chosen[j]:
            return 0
        return sum(1 for e in self.cols[j] if not self.covered[e])

    def is_redundant(self, j):
        return self.remaining_gain(j) == 0

    def add_set(self, j):
        if self.chosen[j]:
            return 0
        self.chosen[j] = True
        self.solution_cost += self.costs[j]
        newly_covered = 0
        for e in self.cols[j]:
            self.cover_count[e] += 1
            if not self.covered[e]:
                self.covered[e] = True
                self.num_covered += 1
                newly_covered += 1
        return newly_covered

    def remove_set(self, j):
        if not self.chosen[j]:
            return
        self.chosen[j] = False
        self.solution_cost -= self.costs[j]
        for e in self.cols[j]:
            self.cover_count[e] -= 1
            if self.cover_count[e] == 0:
                self.covered[e] = False
                self.num_covered -= 1

    def is_complete_cover(self):
        return self.num_covered == self.m

    def solution_sets(self):
        return np.flatnonzero(self.chosen).tolist()

    def __repr__(self):
        return f"SCPInstance(name={self.name}, m={self.m}, n={self.n})"

## 4. Load the example instances

In [5]:
import re

FNAME_RE = re.compile(r"scp([a-zA-Z]|\d)(\d+)", re.IGNORECASE)

def label_from_filename(path):
    match = FNAME_RE.search(path.name)
    if not match:
        return None
    group, num = match.groups()
    return f"{group.upper()}.{num}"

def load_all_instances(data_dir):
    """Load every file in data_dir whose *name* matches the scp<x><n> pattern,
    regardless of extension (.txt, .TXT, no extension, etc.). Skips anything
    that doesn't match (e.g. this notebook itself, README files, folders)."""
    data_dir = Path(data_dir)
    if not data_dir.exists():
        raise FileNotFoundError(
            f"'{data_dir}' does not exist. Set DATA_DIR to the correct path "
            f"(current working directory is {Path.cwd()})."
        )

    instances = {}
    skipped = []
    for path in sorted(data_dir.iterdir()):
        if not path.is_file():
            continue
        label = label_from_filename(path)
        if label is None:
            skipped.append(path.name)
            continue
        data = read_scp_instance(path)
        instances[label] = SCPInstance(data, name=label)

    if skipped:
        print(f"Skipped {len(skipped)} file(s) that didn't match the scp<x><n> pattern: {skipped}")
    return instances

instances = load_all_instances(DATA_DIR)
print(f"Loaded {len(instances)} instance(s): {list(instances.keys())}")


Loaded 42 instance(s): ['4.2', '4.3', '4.4', '4.5', '4.6', '4.7', '4.8', '4.9', '5.1', '5.2', '5.3', '5.4', '5.5', '5.6', '5.7', '5.8', '5.9', '6.1', '6.2', '6.3', '6.4', '6.5', 'A.1', 'A.2', 'A.3', 'A.4', 'A.5', 'B.1', 'B.2', 'B.3', 'B.4', 'B.5', 'C.1', 'C.2', 'C.3', 'C.4', 'C.5', 'D.1', 'D.2', 'D.3', 'D.4', 'D.5']


## 5. Best-known solutions (for later % deviation reporting)

Taken from the values given in the assignment. Keyed by instance label
(matching the naming used for the OR-Library SCP instances, e.g. `scp41`
through `scp410`, `scp51`...`scp59`, `scp61`...`scp65`, `scpa1`...`scpd5`).

In [6]:
BKS = {
    "4.2": 512, "4.3": 516, "4.4": 494, "4.5": 512,
    "4.6": 560, "4.7": 430, "4.8": 492, "4.9": 641,
    "5.1": 253, "5.2": 302, "5.3": 226, "5.4": 242, "5.5": 211,
    "5.6": 213, "5.7": 293, "5.8": 288, "5.9": 279,
    "6.1": 138, "6.2": 146, "6.3": 145, "6.4": 131, "6.5": 161,
    "A.1": 253, "A.2": 252, "A.3": 232, "A.4": 234, "A.5": 236,
    "B.1": 69,  "B.2": 76,  "B.3": 80,  "B.4": 79,  "B.5": 72,
    "C.1": 227, "C.2": 219, "C.3": 243, "C.4": 219, "C.5": 215,
    "D.1": 60,  "D.2": 66,  "D.3": 72,  "D.4": 62,  "D.5": 61,
}

missing = [lbl for lbl in instances if lbl not in BKS]
if missing:
    print("Loaded instances with no BKS entry (ok, just no deviation to report):", missing)

## 6. Sanity check

Quick smoke test: greedily add subsets in index order until the cover is
complete, using only the incremental API above (no heuristic quality claimed
here — this just exercises `add_set`, `remaining_gain`, `is_complete_cover`
before CH1–CH3 are implemented).

In [7]:
def naive_sequential_cover(inst):
    inst.reset()
    for j in range(inst.n):
        if inst.is_complete_cover():
            break
        if not inst.is_redundant(j):
            inst.add_set(j)
    assert inst.is_complete_cover(), "Failed to cover all elements!"
    return inst.solution_cost, len(inst.solution_sets())

for label, inst in instances.items():
    cost, n_sets = naive_sequential_cover(inst)
    bks = BKS.get(label)
    dev = (cost - bks) / bks * 100 if bks else float("nan")
    print(f"{label}: naive cost={cost:.0f}  ({n_sets} sets)  "
          f"BKS={bks}  dev={dev:.1f}%")

4.2: naive cost=716  (97 sets)  BKS=512  dev=39.8%
4.3: naive cost=681  (93 sets)  BKS=516  dev=32.0%
4.4: naive cost=679  (99 sets)  BKS=494  dev=37.4%
4.5: naive cost=717  (99 sets)  BKS=512  dev=40.0%
4.6: naive cost=763  (95 sets)  BKS=560  dev=36.2%
4.7: naive cost=595  (95 sets)  BKS=430  dev=38.4%
4.8: naive cost=642  (89 sets)  BKS=492  dev=30.5%
4.9: naive cost=919  (105 sets)  BKS=641  dev=43.4%
5.1: naive cost=334  (92 sets)  BKS=253  dev=32.0%
5.2: naive cost=457  (102 sets)  BKS=302  dev=51.3%
5.3: naive cost=291  (86 sets)  BKS=226  dev=28.8%
5.4: naive cost=317  (94 sets)  BKS=242  dev=31.0%
5.5: naive cost=291  (96 sets)  BKS=211  dev=37.9%
5.6: naive cost=318  (96 sets)  BKS=213  dev=49.3%
5.7: naive cost=423  (99 sets)  BKS=293  dev=44.4%
5.8: naive cost=422  (101 sets)  BKS=288  dev=46.5%
5.9: naive cost=397  (92 sets)  BKS=279  dev=42.3%
6.1: naive cost=204  (58 sets)  BKS=138  dev=47.8%
6.2: naive cost=232  (60 sets)  BKS=146  dev=58.9%
6.3: naive cost=207  (54 set

---
**Next steps** (not yet implemented in this notebook):

- `CH1`: pick an uncovered element at random, then a random covering set.
- `CH2`, `CH3`: two further constructive rules (e.g. greedy best-cost-per-new-element, and a second variant of your choice).
- `RE`: redundancy elimination on a complete cover (remove sets that have become unnecessary once the cover is complete).
- Run CH1–CH3 with a fixed random seed, with and without RE, on every instance, and report the average % deviation from BKS and the fraction of instances that improved thanks to RE.

## Stochastic Solution (CH1)

This method introduces randomness to avoid getting stuck in local optima. At each step, it identifies an element that still needs coverage and randomly selects an available set to cover it.

Let $U$ be the set of currently uncovered elements, initialized to $\{1, \dots, m\}$. 

Initialize all selection variables to $x_i = 0$ for $i = 1, \dots, n$.   

Randomly select an uncovered element $j \in U$.  Identify the pool of unselected candidate subsets that cover this specific element $j$. 

A subset $i$ is a valid candidate if $x_i = 0$ and $b_{ij} = 1$.   

Randomly pick one subset $i^*$ from this pool and set its selection variable to $x_{i^*} = 1$.   

Update $U$ by removing all elements $k$ that are covered by this new subset (where $b_{i^*k} = 1$). 

Repeat this process until $U$ is empty, satisfying the constraint $\sum_{i=1}^n b_{ij} \ge 1$ for all $j=1, \dots, m$.   

In [8]:
import random

def ch1_stochastic(inst, seed=None):
    if seed is not None:
        random.seed(seed)
    
    inst.reset() 
    
    while not inst.is_complete_cover(): 
        # 1. Pick a random uncovered element
        uncov = inst.uncovered_elements() 
        e = random.choice(uncov)
        
        # 2. Find valid candidate sets covering element 'e'
        # inst.rows[e] gives all sets that cover 'e' 
        candidates = [i for i in inst.rows[e] if not inst.chosen[i]] 
        
        # 3. Pick a random candidate and add it
        # Since 'e' is uncovered, this candidate guarantees a remaining_gain >= 1, 
        # naturally satisfying the non-redundant construction rule .
        best_i = random.choice(candidates)
        inst.add_set(best_i)  
        
    return inst.solution_cost 

## 2. Dynamic Solution (CH2)

This method evaluates subsets based on a cost-effectiveness ratio that changes as elements are covered.

Maintain the set of uncovered elements $U$. 

For each unselected subset $i$ (where $x_i = 0$), calculate the number of currently uncovered elements it can cover: $\sum_{j \in U} b_{ij}$.   

Assign a dynamic score $E_i$ to each subset $i$ by dividing its weight $w_i$ by its current coverage contribution: $E_i = w_i / \sum_{j \in U} b_{ij}$.   Select the subset $i^*$ that minimizes $E_i$ and set $x_{i^*} = 1$.   

Update $U$ by removing the newly covered elements. In the next iteration, the denominator $\sum_{j \in U} b_{ij}$ dynamically updates for the remaining subsets because $U$ has shrunk.   

In [9]:
def ch2_dynamic(inst):
    inst.reset() 
    
    while not inst.is_complete_cover(): 
        best_i = None
        best_score = float('inf')
        
        for i in range(inst.n): 
            # Enforce the non-redundant rule: skip if remaining_gain(j) == 0 
            if not inst.chosen[i] and not inst.is_redundant(i): 
                
                # Score = Cost / Newly Covered Elements 
                score = inst.costs[i] / inst.remaining_gain(i) 
                
                if score < best_score:
                    best_score = score
                    best_i = i
                    
        inst.add_set(best_i)  
        
    return inst.solution_cost 

## 3. Composite Solution (CH3)

## Redundancy Elimination (RE)
Because earlier decisions might become obsolete as subsequent sets overlap with them, we can clean up the final solution. The most effective strategy is to attempt to remove the most expensive sets first to maximize cost savings. 

In [10]:
def apply_redundancy_elimination(inst):
    # Sort the chosen subsets 'i' by cost (descending) to maximize potential savings 
    chosen_subsets = sorted(inst.solution_sets(), key=lambda i: inst.costs[i], reverse=True) 
    
    for i in chosen_subsets:
        # Subset 'i' is redundant if EVERY element 'j' it covers is covered by at least one OTHER subset
        is_redundant = all(inst.cover_count[j] > 1 for j in inst.cols[i]) 
        
        if is_redundant:
            inst.remove_set(i) 
            
    return inst.solution_cost 

## Implementation of solutions

In [16]:
def evaluate_all_instances(instances, bks_dict, seed=42):
    # Dictionaries to store the metrics for each heuristic
    metrics = {
        "CH1 (Stochastic)": {"dev_no_re": [], "dev_re": [], "improved_count": 0},
        "CH2 (Dynamic)":    {"dev_no_re": [], "dev_re": [], "improved_count": 0},
        "CH3 (Composite)":  {"dev_no_re": [], "dev_re": [], "improved_count": 0}
    }
    
    valid_count = 0
    
    # Loop through every instance in your dataset
    for label, inst in instances.items():
        if label not in bks_dict:
            continue # Skip instances if we don't have a known optimal solution to compare against
            
        bks = bks_dict[label]
        valid_count += 1
        
        # --- Run CH1 ---
        cost_ch1_no_re = ch1_stochastic(inst, seed=seed)
        cost_ch1_re = apply_redundancy_elimination(inst)
        
        metrics["CH1 (Stochastic)"]["dev_no_re"].append((cost_ch1_no_re - bks) / bks * 100)
        metrics["CH1 (Stochastic)"]["dev_re"].append((cost_ch1_re - bks) / bks * 100)
        if cost_ch1_re < cost_ch1_no_re:
            metrics["CH1 (Stochastic)"]["improved_count"] += 1
            
        # --- Run CH2 ---
        cost_ch2_no_re = ch2_dynamic(inst)
        cost_ch2_re = apply_redundancy_elimination(inst)
        
        metrics["CH2 (Dynamic)"]["dev_no_re"].append((cost_ch2_no_re - bks) / bks * 100)
        metrics["CH2 (Dynamic)"]["dev_re"].append((cost_ch2_re - bks) / bks * 100)
        if cost_ch2_re < cost_ch2_no_re:
            metrics["CH2 (Dynamic)"]["improved_count"] += 1
            
        # --- Run CH3 ---
        """cost_ch3_no_re = ch3_composite(inst)
        cost_ch3_re = apply_redundancy_elimination(inst)
        
        metrics["CH3 (Composite)"]["dev_no_re"].append((cost_ch3_no_re - bks) / bks * 100)
        metrics["CH3 (Composite)"]["dev_re"].append((cost_ch3_re - bks) / bks * 100)
        if cost_ch3_re < cost_ch3_no_re:
            metrics["CH3 (Composite)"]["improved_count"] += 1"""

        print(f"{label}: ({n_sets} sets) BKS={bks}" f"\nCH1 cost={cost_ch1_no_re:.0f} CH1 cost (post re)={cost_ch1_re:.0f} "
              f"\nCH2 cost={cost_ch2_no_re:.0f} CH2 cost (post re)={cost_ch2_re:.0f} ")
              #f"\nCH3 cost={cost_ch3_no_re:.0f} CH3 cost (post re)={cost_ch3_re:.0f} dev = {}"
              

    # Calculate final averages and print the report
    print(f"Evaluated {valid_count} instances with known BKS.\n")
    
    # Header for the text-based table output in Jupyter
    print(f"{'Heuristic':<20} | {'Avg Dev (No RE)':<15} | {'Avg Dev (With RE)':<17} | {'Fraction Improved by RE'}")
    print("-" * 85)
    
    for name, data in metrics.items():
        avg_dev_no = sum(data["dev_no_re"]) / valid_count
        avg_dev_re = sum(data["dev_re"]) / valid_count
        fraction_imp = data["improved_count"] / valid_count
        
        print(f"{name:<20} | {avg_dev_no:>14.2f}% | {avg_dev_re:>16.2f}% | {data['improved_count']:>5} / {valid_count} ({fraction_imp:.1%})")

# Run the evaluation using the variables already loaded in your notebook[cite: 3]
evaluate_all_instances(instances, BKS, seed=42)

4.2: (70 sets) BKS=512
CH1 cost=3887 CH1 cost (post re)=3106 
CH2 cost=582 CH2 cost (post re)=529 
4.3: (70 sets) BKS=516
CH1 cost=4231 CH1 cost (post re)=3344 
CH2 cost=598 CH2 cost (post re)=537 
4.4: (70 sets) BKS=494
CH1 cost=3642 CH1 cost (post re)=2963 
CH2 cost=548 CH2 cost (post re)=506 
4.5: (70 sets) BKS=512
CH1 cost=4184 CH1 cost (post re)=3209 
CH2 cost=577 CH2 cost (post re)=518 
4.6: (70 sets) BKS=560
CH1 cost=3832 CH1 cost (post re)=2956 
CH2 cost=615 CH2 cost (post re)=594 
4.7: (70 sets) BKS=430
CH1 cost=4034 CH1 cost (post re)=3036 
CH2 cost=476 CH2 cost (post re)=447 
4.8: (70 sets) BKS=492
CH1 cost=4066 CH1 cost (post re)=3004 
CH2 cost=533 CH2 cost (post re)=525 
4.9: (70 sets) BKS=641
CH1 cost=4399 CH1 cost (post re)=3495 
CH2 cost=747 CH2 cost (post re)=665 
5.1: (70 sets) BKS=253
CH1 cost=4243 CH1 cost (post re)=3182 
CH2 cost=289 CH2 cost (post re)=269 
5.2: (70 sets) BKS=302
CH1 cost=3936 CH1 cost (post re)=3251 
CH2 cost=348 CH2 cost (post re)=330 
5.3: (70 s